In [2]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [3]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [3]:
gold = 'GC=F'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
response = request_api.update_commodity_timeseries_data(
    code=gold,
    market=None,
    start=start,
    end=end
)
response

{'result': True}

In [4]:
gold_df = request_api.get_commodity_time_series_data(
    code=gold,
    market=None,
    start=start,
    end=end
)
gold_df.head()

取得件数: 6450


,id,commodity_id,commodity_code,commodity_market,date,open,high,low,close,adj_close,...,rci9,rci26,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1,1,GC=F,None,2000-08-30,273.899994,273.899994,273.899994,273.899994,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,2,1,GC=F,None,2000-08-31,278.299988,278.299988,274.799988,274.799988,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,3,1,GC=F,None,2000-09-01,277.000000,277.000000,277.000000,277.000000,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,4,1,GC=F,None,2000-09-05,275.799988,275.799988,275.799988,275.799988,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,5,1,GC=F,None,2000-09-06,274.200012,274.200012,274.200012,274.200012,None,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [5]:
code = 'LG'
market = 'V'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=code,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [6]:
agmr_timeseries_df = request_api.get_stock_time_series_data(
    code=code,
    market=market,
    start=start,
    end=end
)
agmr_timeseries_df

取得件数: 1026


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1210266,LG,V,2022-04-13,0.380,0.450,0.350,0.350,615000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1210267,LG,V,2022-04-14,0.380,0.400,0.380,0.390,255700,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1210268,LG,V,2022-04-18,0.400,0.400,0.400,0.400,483847,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1210269,LG,V,2022-04-19,0.400,0.400,0.400,0.400,71500,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1210270,LG,V,2022-04-20,0.400,0.400,0.400,0.400,0,0.388,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021,1211287,LG,V,2026-05-08,0.400,0.405,0.380,0.395,845102,0.377,...,0.408862,0.355938,False,NaN,NaN,NaN,NaN,40.0,NaN,False
1022,1211288,LG,V,2026-05-11,0.375,0.400,0.365,0.380,1603871,0.380,...,0.409233,0.357967,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1023,1211289,LG,V,2026-05-12,0.380,0.385,0.360,0.380,1164269,0.384,...,0.409413,0.360587,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1024,1211290,LG,V,2026-05-13,0.380,0.385,0.370,0.380,651521,0.387,...,0.409645,0.361155,True,0.387,NaN,NaN,NaN,NaN,NaN,False


In [7]:
def stock_prices_and_gold_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_gold: pd.DataFrame | None = None,
        df_silver: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # 金価格を統合
    if df_gold is not None:
        df_gold_tmp = df_gold.copy() if df_gold is not None else pd.DataFrame()
        if "date" not in df_gold_tmp.columns:
            df_gold_tmp = df_gold_tmp.reset_index()
        df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
        df_gold_tmp = df_gold_tmp.set_index("date")
        df_gold_tmp = df_gold_tmp.loc[start:end]

    # 銀価格を統合
    if df_silver is not None:
        df_silver_tmp = df_silver.copy() if df_silver is not None else pd.DataFrame()
        if "date" not in df_silver_tmp.columns:
            df_silver_tmp = df_silver_tmp.reset_index()
        df_silver_tmp["date"] = pd.to_datetime(df_silver_tmp["date"])
        df_silver_tmp = df_silver_tmp.set_index("date")
        df_silver_tmp = df_silver_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_gold is not None:
        df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
        df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)
    if df_silver is not None:
        df["MA5_SILVER"] = df_silver_tmp["ma5"].reindex(df.index)
        df["MA25_SILVER"] = df_silver_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- GOLD（右軸） ---
    if df_gold is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_GOLD"],
                name="GOLD_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_GOLD"],
                name="GOLD_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- SILVER（左軸） ---
    if df_silver is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SILVER"],
                name="SILVER_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SILVER"],
                name="SILVER_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [8]:
name = "Lahontan Gold Corp"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime('%Y-%m-%d')
# グラフ領域の作成
fig = stock_prices_and_gold_prices(
    code=code,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_gold=gold_df,
    df_silver=None
)
fig.show()

取得件数: 550


In [9]:
lg_financials_data = request_api.get_corp_financials_data(code=code, market=market)
lg_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=code, market=market)
lg_cash_flow_data = request_api.get_corp_cash_flow_data(code=code, market=market)
lg_earnings_data = request_api.get_corp_earnings_data(code=code, market=market)
lg_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=code, market=market)

GET response: {'detail': 'Corporate finance data not found'}
GET response: {'detail': 'Balance sheet data not found'}
GET response: {'detail': 'Cash flow data not found'}
GET response: {'detail': 'Earnings data not found'}
GET response: {'detail': 'Quarterly earnings data not found'}


In [10]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://lahontangoldcorp.com/wp-content/uploads/2026/04/Lahontan-MDA-December-31-2025-vFinal.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/Lahontan-MDA-December-31-2025-vFinal.pdf.md


'/workspace/data/Lahontan-MDA-December-31-2025-vFinal.pdf.md'

In [11]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://lahontangoldcorp.com/wp-content/uploads/2026/04/Lahontan-Gold-Corp.-Conso-2025-12-RITM30044970-SEDAR.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/Lahontan-Gold-Corp.-Conso-2025-12-RITM30044970-SEDAR.pdf.md


'/workspace/data/Lahontan-Gold-Corp.-Conso-2025-12-RITM30044970-SEDAR.pdf.md'

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="",
    directory_path="/workspace/data",
)
md_file_path

In [ ]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url='',
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/HYMC-2026-Q1-10-Q.pdf.md


'/workspace/data/HYMC-2026-Q1-10-Q.pdf.md'